# GoogLeNet / Inception — Going Deeper with Convolutions

**Primary Citation:**
Szegedy, C., Liu, W., Jia, Y., Sermanet, P., Reed, S., Anguelov, D., ... & Rabinovich, A. (2015). Going deeper with convolutions. *CVPR 2015*. arXiv:1409.4842.

**v3 Citation:**
Szegedy, C., Vanhoucke, V., Ioffe, S., Shlens, J., & Wojna, Z. (2016). Rethinking the inception architecture for computer vision. *CVPR 2016*. arXiv:1512.00567.

**Registered names:** `inception_v1`, `inception_v3`

**Summary:** GoogLeNet introduced the Inception module — applying multiple filter sizes (1×1, 3×3, 5×5) in parallel and concatenating results. This multi-scale approach achieved 6.67% top-5 error at ILSVRC 2014 with only 6.8M parameters (vs AlexNet's 60M).

---

| Model | Depth | Params | ILSVRC Top-5 |
|-------|-------|--------|-------------|
| Inception-v1 | 22 | 6.8M | 6.67% |
| Inception-v3 | 48 | 23.8M | 3.58% |

## Architecture Diagram

```
Input [B, 3, 224, 224]
       │
     Stem (7×7 s2 → MaxPool → 3×3 → MaxPool)
       │  [B, 192, H/8, W/8]
       │
┌──────────────────────────────────────────────────┐
│ Inception Module (parallel branches):           │
│                                                  │
│   ┌──────┐  ┌──────────────┐  ┌───────────────┐  ┌────────────────────┐
│   │ 1×1  │  │ 1×1 → 3×3   │  │ 1×1 → 5×5    │  │ MaxPool → 1×1     │
│   │ Conv │  │ Reduction    │  │ Reduction     │  │ Projection         │
│   └──────┘  └──────────────┘  └───────────────┘  └────────────────────┘
│        └─────────────────┬──────────────────────────────┘
│                          │ Concatenate on channel dim
└──────────────────────────┼──────────────────────────────┘
                           │
       Stage 1: 3a(→256) + 3b(→480) + MaxPool
       Stage 2: 4a..4e (→832) + MaxPool
       Stage 3: 5a(→832) + 5b(→1024) + GlobalAvgPool
                           │
                    pooled_output [B, 1024]
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_facecolor('#0f1117')

# ─── Left: Inception Module detail ───
ax = axes[0]
ax.set_facecolor('#0f1117')
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
ax.text(5, 9.5, 'Inception Module', ha='center', color='white', fontsize=13, fontweight='bold')

branches = [
    ('Branch 1\n1×1 Conv', '#4a9eff', 1.2),
    ('Branch 2\n1×1 → 3×3', '#6bcb77', 3.4),
    ('Branch 3\n1×1 → 5×5', '#ff9f43', 5.8),
    ('Branch 4\nMaxPool → 1×1', '#a29bfe', 8.0),
]

# Input bar
ax.add_patch(FancyBboxPatch((1, 7.8), 8, 0.7, boxstyle='round,pad=0.1',
                             facecolor='#444455', edgecolor='white', linewidth=1.2))
ax.text(5, 8.15, 'Input Feature Map [B, C, H, W]', ha='center', color='white', fontsize=9)

for label, color, cx in branches:
    ax.add_patch(FancyBboxPatch((cx - 0.9, 5.0), 1.8, 2.4,
                                 boxstyle='round,pad=0.1', facecolor=color,
                                 edgecolor='white', linewidth=0.8, alpha=0.85))
    ax.text(cx, 6.2, label, ha='center', va='center', color='white', fontsize=8, fontweight='bold')
    ax.annotate('', xy=(cx, 7.8), xytext=(cx, 7.4),
                arrowprops=dict(arrowstyle='->', color='white', lw=1.2))
    ax.annotate('', xy=(cx, 4.0), xytext=(cx, 5.0),
                arrowprops=dict(arrowstyle='->', color='white', lw=1.2))

# Output concat bar
ax.add_patch(FancyBboxPatch((1, 2.8), 8, 0.9, boxstyle='round,pad=0.1',
                             facecolor='#222233', edgecolor='white', linewidth=1.5))
ax.text(5, 3.25, 'Concatenate  [B, c1+c2+c3+c4, H, W]', ha='center', color='white', fontsize=9, fontweight='bold')

# Connect branches to concat
for _, _, cx in branches:
    ax.plot([cx, cx], [4.0, 3.7], color='white', lw=1, alpha=0.7)
ax.plot([1.2, 8.0], [3.7, 3.7], color='white', lw=1, alpha=0.7)
ax.annotate('', xy=(5, 2.8), xytext=(5, 3.7),
            arrowprops=dict(arrowstyle='->', color='white', lw=1.5))

ax.text(5, 2.1, 'Multiple receptive fields, one unified representation',
        ha='center', color='#aaaaaa', fontsize=8, style='italic')

# ─── Right: GoogLeNet Stage Flow ───
ax2 = axes[1]
ax2.set_facecolor('#0f1117')
ax2.set_xlim(0, 8); ax2.set_ylim(0, 10); ax2.axis('off')
ax2.text(4, 9.5, 'GoogLeNet (Inception-v1) Flow', ha='center', color='white', fontsize=13, fontweight='bold')

flow_blocks = [
    ('Stem\nConv7×7 + Pool\nConv3×3 + Pool', '#555566', 8.2),
    ('Stage 1\n3a(256ch) + 3b(480ch)\n+ MaxPool → H/16', '#4a9eff', 6.7),
    ('Stage 2\n4a→4e (up to 832ch)\n+ MaxPool → H/32', '#6bcb77', 5.0),
    ('Stage 3\n5a(832ch) + 5b(1024ch)', '#ff9f43', 3.3),
    ('GlobalAvgPool\n+ Dropout\npooled [B, 1024]', '#a29bfe', 1.6),
]

for label, color, cy in flow_blocks:
    ax2.add_patch(FancyBboxPatch((1.2, cy - 0.6), 5.6, 1.2,
                                  boxstyle='round,pad=0.1', facecolor=color,
                                  edgecolor='white', linewidth=0.8, alpha=0.85))
    ax2.text(4, cy, label, ha='center', va='center', color='white', fontsize=8, fontweight='bold')

for i in range(len(flow_blocks) - 1):
    y1 = flow_blocks[i][2] - 0.6
    y2 = flow_blocks[i+1][2] + 0.6
    ax2.annotate('', xy=(4, y2), xytext=(4, y1),
                arrowprops=dict(arrowstyle='->', color='white', lw=1.5))

plt.tight_layout()
plt.show()

## Key Innovations

### 1. The Inception Module — Multi-Scale Feature Extraction
Instead of choosing a single filter size, the Inception module applies 1×1, 3×3, and 5×5 convolutions in parallel:
$$\text{Inception}(x) = \text{Concat}[W_{1\times1} * x,\ W_{3\times3} * W_r * x,\ W_{5\times5} * W_r' * x,\ \text{MaxPool} * W_p * x]$$

### 2. Dimensionality Reduction with 1×1 Convolutions
Before the 3×3 and 5×5 convolutions, a 1×1 conv reduces the channel count:
$$\text{Cost without reduction: } H \cdot W \cdot C_{in} \cdot k^2 \cdot C_{out}$$
$$\text{Cost with reduction: } H \cdot W \cdot C_{in} \cdot C_r + H \cdot W \cdot C_r \cdot k^2 \cdot C_{out}$$
At $C_r \ll C_{in}$, this gives ~10× parameter reduction for the 5×5 branch.

### 3. Inception-v3: Factorised Convolutions
Any $n \times n$ conv can be approximated by $1 \times n$ followed by $n \times 1$:
$$\text{Params}(n \times n) = n^2 \quad \text{vs} \quad \text{Params}(1 \times n + n \times 1) = 2n$$
For 7×7: saves 49-14=35 parameters per filter, a 28.5% reduction.

### 4. No Large FC Head
GoogLeNet uses Global Average Pooling (not FC layers) to produce the final embedding — eliminating 102M parameters that dominate AlexNet/VGG.

## Code Walkthrough

Implemented in `hale_core/nn/vision/cnn/inception.py`.

**InceptionModuleV1 — the core building block:**
```python
class InceptionModuleV1(nn.Module):
    def __init__(self, in_ch, c1, c2_r, c2, c3_r, c3, c4):
        self.branch1 = ConvBnRelu(in_ch, c1, kernel_size=1)
        self.branch2 = nn.Sequential(
            ConvBnRelu(in_ch, c2_r, kernel_size=1),
            ConvBnRelu(c2_r, c2,   kernel_size=3, padding=1),
        )
        self.branch3 = nn.Sequential(
            ConvBnRelu(in_ch, c3_r, kernel_size=1),
            ConvBnRelu(c3_r, c3,   kernel_size=5, padding=2),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            ConvBnRelu(in_ch, c4, kernel_size=1),
        )

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x),
                          self.branch3(x), self.branch4(x)], dim=1)
```

**GoogLeNet stage 2 (9 Inception modules including 4a–4e):**
The auxiliary classifiers from the original paper are omitted — they were a training trick to inject gradient early, but modern BN + Adam makes them unnecessary.

In [ ]:
import torch
import sys
sys.path.insert(0, '../../src')

from hale_core.nn.vision.registry import build_vision_backbone

for name, size in [('inception_v1', 224), ('inception_v3', 299)]:
    bb = build_vision_backbone(name, img_size=size)
    bb.eval()
    x = torch.randn(1, 3, size, size)
    with torch.no_grad():
        out = bb(x)
    print(f'{name} (input {size}×{size}):')
    print(f'  pooled_output: {tuple(out.pooled_output.shape)}')
    print(f'  feature_maps:  {[tuple(f.shape) for f in out.feature_maps]}')
    print(f'  params: {bb.num_parameters()/1e6:.1f}M')

## References

| Paper | Venue | Key Contribution |
|-------|-------|------------------|
| Szegedy et al. (2015). Going Deeper with Convolutions. | CVPR 2015 | Inception-v1 / GoogLeNet — parallel multi-scale branches |
| Szegedy et al. (2016). Rethinking the Inception Architecture. | CVPR 2016 | Inception-v3 — factorised convolutions |
| Szegedy et al. (2017). Inception-v4, Inception-ResNet. | AAAI 2017 | Inception + residual connections |
| Lin et al. (2014). Network in Network. | ICLR 2014 | 1×1 convolutions as dimensionality reduction |
| Ioffe & Szegedy (2015). Batch Normalisation. | ICML 2015 | Enabled removing auxiliary classifiers |

**Full Citations:**
> Szegedy, C., Liu, W., Jia, Y., Sermanet, P., Reed, S., Anguelov, D., Erhan, D., Vanhoucke, V., & Rabinovich, A. (2015). Going deeper with convolutions. In *Proceedings of CVPR 2015* (pp. 1–9). arXiv:1409.4842.
>
> Szegedy, C., Vanhoucke, V., Ioffe, S., Shlens, J., & Wojna, Z. (2016). Rethinking the inception architecture for computer vision. In *Proceedings of CVPR 2016* (pp. 2818–2826). arXiv:1512.00567.